In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Both sources belong to the same development pool; mentor test data are separate.
DATASET_ROOTS = [PROJECT_ROOT / "dataset", PROJECT_ROOT / "dataset_extra"]
for root in DATASET_ROOTS:
    if not root.is_dir():
        raise FileNotFoundError(f"Missing dataset source: {root}")
DATASET_ROOT = DATASET_ROOTS[0]  # Only for the introductory single-image example.
print("Dataset sources:", [root.name for root in DATASET_ROOTS])

In [ ]:
for DATASET_ROOT in DATASET_ROOTS:
    print("Source:", DATASET_ROOT.name)
    for split_name in ["train", "test", "unclean"]:
        split_path = DATASET_ROOT / split_name
        if not split_path.exists():
            continue
    
        print(f"{split_name}: exists={split_path.is_dir()}")

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
class_counts = []
for DATASET_ROOT in DATASET_ROOTS:
    print("Source:", DATASET_ROOT.name)
    for split_name in ["train", "test", "unclean"]:
        split_path = DATASET_ROOT / split_name
        if not split_path.exists():
            continue
        total_images = 0
    
        print(f"\nSplit: {split_name}")
    
        for class_path in sorted(split_path.iterdir()):
            if not class_path.is_dir():
                continue
    
            image_paths = [
                path
                for path in class_path.iterdir()
                if path.is_file()
                and path.suffix.lower() in IMAGE_EXTENSIONS
            ]
    
            count = len(image_paths)
            total_images += count
    
            class_counts.append({
                "split": split_name,
                "class_name": class_path.name,
                "count": count,
            })
    
            print(f"  {class_path.name}: {count}")
    
        print(f"Total: {total_images}")

In [ ]:
from PIL import Image

sample_path = next(
    path
    for path in sorted((DATASET_ROOT / "train" / "ambulance").iterdir())
    if path.is_file()
    and path.suffix.lower() in IMAGE_EXTENSIONS
)

with Image.open(sample_path) as image:
    image.load()

    print("File:", sample_path.name)
    print("Format:", image.format)
    print("Size:", image.size)
    print("Color mode:", image.mode)

In [ ]:
def inspect_image(image_path):
    result = {
        "path": str(image_path),
        "width": None,
        "height": None,
        "format": None,
        "mode": None,
        "readable": False,
        "error": None,
    }

    try:
        with Image.open(image_path) as image:
            image.load()

            result["width"] = image.width
            result["height"] = image.height
            result["format"] = image.format
            result["mode"] = image.mode
            result["readable"] = True

    except (OSError, ValueError) as error:
        result["error"] = str(error)

    return result

In [ ]:
sample_info = inspect_image(sample_path)
print(sample_info)

In [ ]:
image_records = []
for DATASET_ROOT in DATASET_ROOTS:
    print("Source:", DATASET_ROOT.name)
    for split_name in ["train", "test", "unclean"]:
        split_path = DATASET_ROOT / split_name
        if not split_path.exists():
            continue
    
        for class_path in sorted(split_path.iterdir()):
            if not class_path.is_dir():
                continue
    
            for image_path in sorted(class_path.iterdir()):
                if not image_path.is_file():
                    continue
    
                if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
    
                record = inspect_image(image_path)
    
                record["source_dataset"] = DATASET_ROOT.name
                record["source_split"] = split_name
                record["label"] = class_path.name
    
                image_records.append(record)
    

print("Inspected images:", len(image_records))
unreadable_records = [r for r in image_records if not r["readable"]]
print("Unreadable images:", len(unreadable_records))

In [ ]:
readable_records = [
    record
    for record in image_records
    if record["readable"]
]

for record in readable_records:
    record["short_side"] = min(record["width"], record["height"])
    record["aspect_ratio"] = record["width"] / record["height"]

for split_name in ["train", "test", "unclean"]:
    split_records = [
        record
        for record in readable_records
        if record["source_split"] == split_name
    ]

    if not split_records:
        continue

    widths = [record["width"] for record in split_records]
    heights = [record["height"] for record in split_records]
    ratios = [record["aspect_ratio"] for record in split_records]

    print(f"\nSplit: {split_name}")
    print(f"Width range: {min(widths)} to {max(widths)}")
    print(f"Height range: {min(heights)} to {max(heights)}")
    print(f"Aspect ratio range: {min(ratios):.2f} to {max(ratios):.2f}")

In [ ]:
from IPython.display import display

train_records = [
    record
    for record in readable_records
    if record["source_split"] == "train"
]

smallest_records = sorted(
    train_records,
    key=lambda record: record["short_side"],
)[:6]

smallest_paths = {
    record["path"]
    for record in smallest_records
}

remaining_records = [
    record
    for record in train_records
    if record["path"] not in smallest_paths
]

most_elongated_records = sorted(
    remaining_records,
    key=lambda record: max(
        record["aspect_ratio"],
        1 / record["aspect_ratio"],
    ),
    reverse=True,
)[:6]

for title, records in [
    ("Smallest images", smallest_records),
    ("Most elongated images", most_elongated_records),
]:
    print(f"\n{title}")

    for record in records:
        print(
            f"Class: {record['label']} | "
            f"Size: {record['width']}x{record['height']} | "
            f"Ratio: {record['aspect_ratio']:.2f}"
        )
        print("Path:", record["path"])

        with Image.open(record["path"]) as image:
            preview = image.convert("RGB")
            preview.thumbnail((320, 320))
            display(preview)

In [ ]:
import hashlib

def compute_image_hash(image_path):
    with Image.open(image_path) as image:
        rgb_image = image.convert("RGB")

        hasher = hashlib.sha256()
        hasher.update(str(rgb_image.size).encode("utf-8"))
        hasher.update(rgb_image.tobytes())

        return hasher.hexdigest()

In [ ]:
sample_hash = compute_image_hash(sample_path)
print(sample_hash)

In [ ]:
hash_groups = {}

for record in readable_records:
    image_hash = compute_image_hash(record["path"])
    record["content_hash"] = image_hash

    if image_hash not in hash_groups:
        hash_groups[image_hash] = []

    hash_groups[image_hash].append(record)

duplicate_groups = [
    group
    for group in hash_groups.values()
    if len(group) > 1
]

print("Images checked:", len(readable_records))
print("Unique image contents:", len(hash_groups))
print("Duplicate groups:", len(duplicate_groups))


extra_copies = sum(
    len(group) - 1
    for group in duplicate_groups
)

print("Extra duplicate copies:", extra_copies)

In [ ]:
cross_split_groups = []
label_conflict_groups = []

for group_number, group in enumerate(duplicate_groups, start=1):
    splits = {record["source_split"] for record in group}
    labels = {record["label"] for record in group}

    if len(splits) > 1:
        cross_split_groups.append(group)

    if len(labels) > 1:
        label_conflict_groups.append(group)

    print(f"\nGroup {group_number}")

    for record in group:
        print(
            f"  {record['source_split']}/{record['label']}"
            f" ﷿﷿﷿ {record['path']}"
        )

    print("Across splits:", len(splits) > 1)
    print("Label conflict:", len(labels) > 1)

print("\nDuplicate groups across splits:", len(cross_split_groups))
print("Groups with conflicting labels:", len(label_conflict_groups))

In [ ]:
for group in label_conflict_groups:
    print("Same image, different labels:")

    for record in group:
        print(
            f"  split={record['source_split']}, "
            f"label={record['label']}, "
            f"path={record['path']}"
        )

    with Image.open(group[0]["path"]) as image:
        preview = image.convert("RGB")
        preview.thumbnail((400, 400))
        display(preview)

In [ ]:
# Refresh the private inventory without losing decisions for unchanged images.
import csv
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.data_split import build_manifest

review_file = PROJECT_ROOT / "reports/data_review_decisions.csv"
fields = ["path", "original_label", "content_hash", "audit_status",
          "decision", "reviewed_label", "reason"]
previous = {}
if review_file.exists():
    with review_file.open(newline="", encoding="utf-8") as file:
        previous = {
            (row["path"], row["content_hash"]): row
            for row in csv.DictReader(file)
        }
review_rows = []
for record in readable_records:
    path = Path(record["path"]).relative_to(PROJECT_ROOT).as_posix()
    key = (path, record["content_hash"])
    review_rows.append(previous.get(key, {
        "path": path, "original_label": record["label"],
        "content_hash": record["content_hash"], "audit_status": "readable",
        "decision": "pending", "reviewed_label": "", "reason": "",
    }))
review_file.parent.mkdir(parents=True, exist_ok=True)
with review_file.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    writer.writerows(review_rows)

split_manifest, split_summary = build_manifest()
print("Unique eligible images:", split_summary["eligible_unique_images"])
print("Pending-label images in validation:", split_summary["provisional_validation_images"])
print("Nissan target class:", split_manifest["neysan_target_label"])


In [ ]:
# Stratification preserves proportions within class and source, not equal class sizes.
from collections import Counter
for name in ("training", "validation"):
    print(name, "classes:", split_summary[f"{name}_by_class"])
    print(name, "sources:", split_summary[f"{name}_by_source"])


In [ ]:
# The one split is created before either model is trained.
samples_by_path = {row["path"]: row for row in split_manifest["samples"]}
training_records = [samples_by_path[path] for path in split_manifest["training_paths"]]
validation_records = [samples_by_path[path] for path in split_manifest["validation_paths"]]
print("Training:", len(training_records))
print("Validation:", len(validation_records))
print("Validation unclean:", sum(row["source"] == "unclean" for row in validation_records))
print("Validation Nissan:", sum(row["source"] == "neysan" for row in validation_records))


In [ ]:
# Exact and conservatively detected near-duplicate groups stay on one side.
training_hashes = {row["content_hash"] for row in training_records}
validation_hashes = {row["content_hash"] for row in validation_records}
training_groups = {row["near_duplicate_group"] for row in training_records}
validation_groups = {row["near_duplicate_group"] for row in validation_records}
assert training_hashes.isdisjoint(validation_hashes)
assert training_groups.isdisjoint(validation_groups)
assert len(training_records) + len(validation_records) == len(split_manifest["samples"])
print("Exact duplicate overlap:", len(training_hashes & validation_hashes))
print("Near-duplicate group overlap:", len(training_groups & validation_groups))


In [ ]:
# These manifests contain private paths and stay outside Git.
import json
split_file = PROJECT_ROOT / "reports/base_split.json"
summary_file = PROJECT_ROOT / "reports/data_summary.json"
split_file.parent.mkdir(parents=True, exist_ok=True)
split_file.write_text(json.dumps(split_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
summary_file.write_text(json.dumps(split_summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Saved private split:", split_file.name)
print("Fingerprint:", split_manifest["fingerprint"][:12])


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# review_folders = ["train/vanet", "unclean/vanet", "unclean/neysan", "train/kamyun", "train/kamyunet"]
# IMAGES_PER_PAGE = 25
# for DATASET_ROOT in DATASET_ROOTS:
#     for relative_folder in review_folders:
#         image_paths = sorted(
#             (DATASET_ROOT / relative_folder).glob("*.jpg")
#         )
#         print(f"{relative_folder}: {len(image_paths)} images")
#
#         for start in range(0, len(image_paths), IMAGES_PER_PAGE):
#             page_paths = image_paths[start:start + IMAGES_PER_PAGE]
#             fig, axes = plt.subplots(5, 5, figsize=(16, 17))
#             axes = axes.ravel()
#
#             for offset, image_path in enumerate(page_paths):
#                 ax = axes[offset]
#                 with Image.open(image_path) as image:
#                     ax.imshow(image.convert("RGB"))
#                 ax.set_title(
#                     f"{start + offset + 1}: {image_path.name}",
#                     fontsize=10,
#                 )
#
#             for ax in axes:
#                 ax.axis("off")
#
#             fig.suptitle(
#                 f"{relative_folder} | "
#                 f"{start + 1}-{start + len(page_paths)} / {len(image_paths)}"
#             )
#             plt.tight_layout(rect=(0, 0, 1, 0.97))
#             plt.show()
#             plt.close(fig)

## یک سیاست داده برای کل پروژه

هر دو دیتاست از ابتدا وارد یک مخزن توسعه می‌شوند. پوشهٔ `test` قدیمی هم مجاز است؛ تست نهایی منتورها جدا و در دسترس ما نیست. تصویرهای `unclean` با برچسب فعلی **موقت** محسوب می‌شوند تا زمانی که تصمیم `accept`، `exclude` یا `relabel` در فایل خصوصی بازبینی ثبت شود. نیسان به‌عنوان زیرنوع `vanet` وارد مسئلهٔ هشت‌کلاسه می‌شود؛ در manifest برچسب اصلی آن نیز حفظ می‌شود تا جداگانه ارزیابی‌اش کنیم.

ابتدا تکرارهای دقیق یک نماینده می‌گیرند و تصاویر خیلی شبیه در یک گروه می‌مانند. سپس یک تقسیم حدود ۸۰/۲۰ با seed ثابت می‌سازیم که نسبت هر **کلاس و منبع** را تا حد امکان نگه دارد. `stratify` به معنی برابر کردن تعداد کلاس‌ها نیست. شرط گروه‌بندی ممکن است باعث شود تعداد دقیق validation چند تصویر با ۲۰٪ فرق کند. تمام مدل‌ها همین یک manifest را می‌خوانند؛ هیچ فهرست داده‌ای بعداً به آموزش چسبانده نمی‌شود.

validation هم تصویرهای منبع اصلی، هم `unclean` هشت‌کلاسه و هم نیسان دارد. چون برچسب‌های بسیاری از `unclean` هنوز بررسی دستی نشده‌اند، دقت این بخش را باید با این محدودیت تفسیر کرد. هیچ فایل خامی جابه‌جا یا منتشر نمی‌شود. با عوض شدن تقسیم، تمام اعداد و checkpointهای قبلی تاریخی‌اند و باید مدل‌ها دوباره آموزش ببینند.
